In [1]:
import pandas as pd

# CSV -> DataFrame 로딩
df = pd.read_csv("titanic_train.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [2]:
# 조건 필터링 (불린 인덱싱): 60세 초과 승객 일부 확인
(df.loc[df["Age"] > 60, ["Name", "Age", "Pclass", "Survived"]].head())

,Name,Age,Pclass,Survived
33,"Wheadon, Mr. Edward H",66.0,2,0
54,"Ostby, Mr. Engelhart Cornelius",65.0,1,0
96,"Goldschmidt, Mr. George B",71.0,1,0
116,"Connors, Mr. Patrick",70.5,3,0
170,"Van der hoef, Mr. Wyckoff",61.0,1,0


In [3]:
# 기본 확인 (크기/컬럼/결측치)
print("shape:", df.shape)
print("\n컬럼:", df.columns.tolist())
print("\n결측치 개수(상위 6개):")
(df.isna().sum().sort_values(ascending=False).head(6))

shape: (891, 12)

컬럼: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

결측치 개수(상위 6개):


Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
dtype: int64

In [4]:
# 컬럼 선택 / 일부 행 확인
df[["Survived", "Pclass"]].head()

,Survived,Pclass
0,0,3
1,1,1
2,1,3
3,1,1
4,0,3


In [ ]:
# 파생 컬럼 생성 
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

df[["SibSp", "Parch", "FamilySize", "IsAlone"]].head()

,SibSp,Parch,FamilySize,IsAlone
0,1,0,2,0
1,1,0,2,0
2,0,0,1,1
3,1,0,2,0
4,0,0,1,1


In [6]:
# 그룹화/집계: Pclass별 생존율/평균 요금/평균 나이/혼자 탑승 비율
summary = (
    df.groupby("Pclass")
    .agg(
        n=("PassengerId", "count"),
        survived_rate=("Survived", "mean"),
        avg_fare=("Fare", "mean"),
        avg_age=("Age", "mean"),
        alone_rate=("IsAlone", "mean"),
    )
    .sort_index()
)
summary

,n,survived_rate,avg_fare,avg_age,alone_rate
Pclass,,,,,
1,216,0.629630,84.154687,38.233441,0.504630
2,184,0.472826,20.662183,29.877630,0.565217
3,491,0.242363,13.675550,25.140620,0.659878


In [7]:
# 정렬: 요금(Fare)이 비싼 순으로 상위 5명
(df.sort_values("Fare", ascending=False)[["Name", "Fare", "Pclass", "Survived"]].head())

,Name,Fare,Pclass,Survived
679,"Cardeza, Mr. Thomas Drake Martinez",512.3292,1,1
258,"Ward, Miss. Anna",512.3292,1,1
737,"Lesurer, Mr. Gustave J",512.3292,1,1
88,"Fortune, Miss. Mabel Helen",263.0000,1,1
438,"Fortune, Mr. Mark",263.0000,1,0


In [8]:
# 결측치 처리(간단 버전)
# - Age: 중앙값
# - Embarked: 최빈값
# - Cabin: 'Unknown'
df["Age"] = df["Age"].fillna(df["Age"].median())
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode(dropna=True)[0])
df["Cabin"] = df["Cabin"].fillna("Unknown")

# 처리 후 결측치 개수 확인
(df.isna().sum().sort_values(ascending=False).head(6))

PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
dtype: int64